In [6]:
# cell-01 – Importy

import subprocess
import numpy as np
import matplotlib.pyplot as plt
import os
from ultralytics import YOLO
from PIL import Image
from scipy.ndimage import binary_fill_holes, label as ndlabel
import tempfile

In [7]:
# cell-02 – Konfiguracja: ścieżki do modelu i zdjęć

MODEL_PATH = "/home/marcin/kod/flir-test/data-flir-test/runs/flir-hand-52/weights/best.pt"
IMAGE_A = "/home/marcin/kod/flir-test/data-flir-test/FLIR0146.jpg"
IMAGE_B = "/home/marcin/kod/flir-test/data-flir-test/FLIR0147.jpg"

In [8]:
# cell-03 – Funkcja ekstrakcji macierzy temperatur z pliku FLIR (wzór Plancka)

def raw_from_temp(T_celsius, R1, R2, B, F, O):
    """Konwertuje temperaturę [°C] na surowy sygnał RAW (odwrotność wzoru Plancka)."""
    T_K = T_celsius + 273.15
    return R1 / (R2 * (np.exp(B / T_K) - F)) - O


def get_flir_data_robust(file_path):
    try:
        # 1. Tworzymy TIFF z surowych danych termicznych
        generated_tiff = file_path.replace(".jpg", ".tiff")
        subprocess.run(["exiftool", "-b", "-RawThermalImage", file_path, "-w!", "tiff"], check=True)

        # Wczytujemy macierz RAW
        raw_matrix = plt.imread(generated_tiff).astype(float)

        # 2. Pobieramy metadane
        cmd_meta = ["exiftool", "-s", "-Planck*", "-*R1", "-*R2", "-*B", "-*F", "-*O",
                    "-Emissivity", "-ReflectedApparentTemperature", "-AtmosphericTemperature",
                    file_path]
        meta_out = subprocess.check_output(cmd_meta).decode()

        params = {}
        for line in meta_out.split('\n'):
            if ':' in line:
                key, val = line.split(':', 1)
                clean_key = key.strip().replace('Planck', '')
                try:
                    params[clean_key] = float(val.strip().split()[0])
                except (ValueError, IndexError):
                    continue

        print(f"Znalezione parametry: {list(params.keys())}")

        # 3. Sprawdzamy czy mamy komplet stałych Plancka
        required = ['R1', 'R2', 'B', 'F', 'O']
        if all(k in params for k in required):
            R1, R2, B, F, O = params['R1'], params['R2'], params['B'], params['F'], params['O']

            emissivity = params.get('Emissivity', 1.0)
            T_refl     = params.get('ReflectedApparentTemperature', 20.0)
            T_atm      = params.get('AtmosphericTemperature', 23.0)

            print(f"Emisyjność: {emissivity}, T_odbicia: {T_refl}°C, T_atm: {T_atm}°C")

            S_refl = raw_from_temp(T_refl, R1, R2, B, F, O)
            S_atm  = raw_from_temp(T_atm,  R1, R2, B, F, O)

            S_obj = (raw_matrix - (1 - emissivity) * S_refl) / emissivity

            denominator = np.clip(R2 * (S_obj + O), 1e-10, None)
            temp_c = B / np.log(R1 / denominator + F) - 273.15

        else:
            print("⚠️ Brak stałych Plancka w metadanych! Wyświetlam surowy sygnał RAW.")
            temp_c = raw_matrix

        # Sprzątanie
        if os.path.exists(generated_tiff):
            os.remove(generated_tiff)

        return temp_c

    except Exception as e:
        print(f"Błąd: {e}")
        return None

In [ ]:
# cell-04 – Funkcje pomocnicze YOLO: rendering + segmentacja + statystyki

def thermal_to_inferno_png(temp_matrix, pad=20):
    """
    Renderuje macierz temperatur do obrazu inferno BEZ nakładek,
    z czarną ramką (padding) żeby YOLO nie ucinał obiektów przy krawędzi.
    Zwraca: ścieżkę PNG, pad, szerokość i wysokość paddowanego obrazu.
    """
    fig = plt.figure(frameon=False)
    ax  = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(temp_matrix, cmap='inferno', aspect='auto')

    tmp_path = tempfile.mktemp(suffix=".png")
    fig.savefig(tmp_path, dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    img = Image.open(tmp_path)
    orig_w, orig_h = img.size
    padded_w = orig_w + 2 * pad
    padded_h = orig_h + 2 * pad
    padded = Image.new("RGB", (padded_w, padded_h), (0, 0, 0))
    padded.paste(img, (pad, pad))
    padded.save(tmp_path)

    return tmp_path, pad, padded_w, padded_h


def hand_stats_from_pair(flir_path, temp_matrix, model):
    tmp_png, pad, padded_w, padded_h = thermal_to_inferno_png(temp_matrix)

    try:
        results = model(tmp_png, verbose=False)[0]
    finally:
        os.remove(tmp_png)

    if results.masks is None or len(results.masks) == 0:
        print(f"  [!] Brak detekcji dłoni: {flir_path}")
        return None, None

    best_idx   = int(results.boxes.conf.argmax())
    confidence = float(results.boxes.conf[best_idx])

    mask_np = results.masks.data[best_idx].cpu().numpy().astype(np.uint8)
    img_h, img_w = mask_np.shape

    py = int(round(pad * img_h / padded_h))
    px = int(round(pad * img_w / padded_w))
    mask_np = mask_np[py:img_h - py, px:img_w - px]

    thermal_h, thermal_w = temp_matrix.shape
    crop_h, crop_w = mask_np.shape

    if (crop_h, crop_w) != (thermal_h, thermal_w):
        mask_pil = Image.fromarray(mask_np * 255)
        mask_pil = mask_pil.resize((thermal_w, thermal_h), Image.NEAREST)
        mask_np  = (np.array(mask_pil) > 127).astype(np.uint8)

    mask_np = binary_fill_holes(mask_np).astype(np.uint8)

    labeled, num_features = ndlabel(mask_np)
    if num_features > 1:
        sizes   = [(labeled == i).sum() for i in range(1, num_features + 1)]
        largest = np.argmax(sizes) + 1
        mask_np = (labeled == largest).astype(np.uint8)

    hand_pixels = temp_matrix[mask_np == 1]

    if len(hand_pixels) == 0:
        print(f"  [!] Maska pusta po resize: {flir_path}")
        return None, None

    stats = {
        "confidence": confidence,
        "n_pixels":   len(hand_pixels),
        "mean_C":     float(np.mean(hand_pixels)),
        "median_C":   float(np.median(hand_pixels)),
        "min_C":      float(np.min(hand_pixels)),
        "max_C":      float(np.max(hand_pixels)),
        "std_C":      float(np.std(hand_pixels)),
    }
    return stats, mask_np

Przetwarzanie obrazu A...


NameError: name 'process_image' is not defined

In [ ]:
# cell-05 – Funkcja wyciągająca statystyki (poprawione nazwy zmiennych)
def get_hand_stats(temp_map, mask, confidence):
    # Wybieramy tylko piksele należące do maski
    hand_pixels = temp_map[mask == 1]
    
    if len(hand_pixels) == 0:
        return None
    
    return {
        "mean_C": np.mean(hand_pixels),
        "median_C": np.median(hand_pixels),
        "min_C": np.min(hand_pixels),
        "max_C": np.max(hand_pixels),
        "p5_C": np.percentile(hand_pixels, 5),   
        "p95_C": np.percentile(hand_pixels, 95), 
        "std_C": np.std(hand_pixels),
        "n_pixels": len(hand_pixels),
        "confidence": confidence
    }

# Ponowne przeliczenie statystyk - UŻYWAMY stats_A['confidence'] jeśli już istniały 
# lub zmiennych z Twojego kodu (prawdopodobnie confidence_A / confidence_B)
stats_A = get_hand_stats(temp_A, mask_A, stats_A['confidence'])
stats_B = get_hand_stats(temp_B, mask_B, stats_B['confidence'])

In [ ]:
# cell-06 – Tabela z poprawionym wyrównaniem (rozdzielone wcięcia)
line_width = 60
print("\n" + "═" * line_width)
print(f"{'Parametr':<20} {'Zdjęcie A':>15} {'Zdjęcie B':>15}  {'Δ (B-A)':>8}")
print("═" * line_width)

main_metrics = [
    ("min_C", "T min", ""),
    ("p5_C", "T P5 (5%)", "  "),    # Wcięcie jako osobny parametr
    ("max_C", "T max", ""),
    ("p95_C", "T P95 (95%)", "  "), # Wcięcie jako osobny parametr
    ("mean_C", "T mean", ""),
    ("median_C", "T median", "")
]

for key, label, indent in main_metrics:
    vA, vB = stats_A[key], stats_B[key]
    delta  = vB - vA
    sign   = "+" if delta >= 0 else ""
    
    # Składamy nazwę z wcięciem, ale dbamy by całość miała stałą szerokość 20
    full_label = f"{indent}{label}"
    print(f"{full_label:<20} {vA:>12.1f}°C {vB:>14.1f}°C  {sign}{delta:>7.1f}°C")

# T std
vA_std, vB_std = stats_A["std_C"], stats_B["std_C"]
str_std_A = f"±{vA_std:.1f}°C"
str_std_B = f"±{vB_std:.1f}°C"
print(f"{'T std':<20} {str_std_A:>14} {str_std_B:>15}  {' - ':>9}")

print("═" * line_width)

# Parametry techniczne
# Używamy identycznego wyrównania (12, 14, 15) jak wyżej dla liczb
print(f"{'Confidence':<20} {stats_A['confidence']:>14.3f} {stats_B['confidence']:>15.3f}")
print(f"{'Pixels':<20} {stats_A['n_pixels']:>14d} {stats_B['n_pixels']:>15d}")

print("═" * line_width)

In [ ]:
# cell-07 – Wizualizacja 2×3: termogram | maska | temperatura dłoni
%matplotlib inline

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Dane do pętli
plot_data = [
    (temp_A, mask_A, stats_A, f"Image A | mean={stats_A['mean_C']:.2f}°C"),
    (temp_B, mask_B, stats_B, f"Image B | mean={stats_B['mean_C']:.2f}°C"),
]

for row, (temp, mask, stats, label) in enumerate(plot_data):
    # --- Column 1: Full thermal image ---
    im0 = axes[row, 0].imshow(temp, cmap='inferno')
    axes[row, 0].set_title(f"{label}\nFull thermal image")
    axes[row, 0].axis('off') 
    
    # Pad=0.03 daje lekki odstęp paska od obrazka, by nie był "przyklejony"
    cbar0 = fig.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.03)
    cbar0.set_label("°C", rotation=0, labelpad=15, va='bottom')

    # --- Column 2: YOLO mask ---
    axes[row, 1].imshow(mask, cmap='gray')
    axes[row, 1].set_title(f"YOLO mask (confidence: {stats['confidence']:.1%})\n"
                            f"Hand pixels: {stats['n_pixels']:,}")
    axes[row, 1].axis('off')

    # --- Column 3: Hand temperature ---
    masked_temp = np.where(mask == 1, temp, np.nan)
    im2 = axes[row, 2].imshow(masked_temp, cmap='inferno')
    axes[row, 2].set_title(f"Hand temperature\n"
                            f"min={stats['min_C']:.2f}°C max={stats['max_C']:.2f}°C std=±{stats['std_C']:.2f}°C")
    axes[row, 2].axis('off')
    
    cbar2 = fig.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.03)
    cbar2.set_label("°C", rotation=0, labelpad=15, va='bottom')

# Zmieniamy wspace na 0.2 (ok. 1/3 mniejszy niż standardowy auto-layout)
# hspace=0.3 zostawiamy dla pionowego oddechu między A i B
fig.subplots_adjust(wspace=0.2, hspace=0.3, left=0.05, right=0.95, top=0.95, bottom=0.05)

plt.savefig('/home/marcin/kod/flir-test/data-flir-test/wynik_porownania.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: Columns tightened by ~1/3 with safe label margins.")

In [ ]:
# cell-08 – Alternatywna wizualizacja: 3 wiersze x 2 kolumny (spójne nagłówki)
%matplotlib inline

# Konfiguracja siatki
fig, axes = plt.subplots(3, 2, figsize=(12, 14))

# Dane do pętli
plot_data = [
    (temp_A, mask_A, stats_A, "Image A"),
    (temp_B, mask_B, stats_B, "Image B")
]

# --- WIERSZ 0: Full thermal images ---
for col, (temp, mask, stats, label) in enumerate(plot_data):
    im = axes[0, col].imshow(temp, cmap='inferno')
    # Spójna struktura: Nazwa i typ w 1. linii, T mean w 2. linii
    axes[0, col].set_title(f"{label} | Full thermal image\nT mean={stats['mean_C']:.1f}°C")
    axes[0, col].axis('off')
    cbar = fig.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)
    cbar.set_label("°C", rotation=0, labelpad=15, va='bottom')

# --- WIERSZ 1: YOLO masks ---
for col, (temp, mask, stats, label) in enumerate(plot_data):
    axes[1, col].imshow(mask, cmap='gray')
    # Spójna struktura: Nazwa i typ w 1. linii, Confidence i Pixels w 2. linii
    axes[1, col].set_title(f"{label} | YOLO mask\n"
                           f"Confidence: {stats['confidence']:.1%} | Pixels: {stats['n_pixels']:d}")
    axes[1, col].axis('off')

# --- WIERSZ 2: Hand temperature (Only hand) ---
for col, (temp, mask, stats, label) in enumerate(plot_data):
    masked_temp = np.where(mask == 1, temp, np.nan)
    im = axes[2, col].imshow(masked_temp, cmap='inferno')
    # Spójna struktura: Nazwa i typ w 1. linii, T min i T max w 2. linii
    axes[2, col].set_title(f"{label} | Hand stats\n"
                           f"T min={stats['min_C']:.1f}°C  T max={stats['max_C']:.1f}°C")
    axes[2, col].axis('off')
    cbar = fig.colorbar(im, ax=axes[2, col], fraction=0.046, pad=0.04)
    cbar.set_label("°C", rotation=0, labelpad=15, va='bottom')

# Ustawienie ciasnych odstępów między wierszami (hspace=0.15)
fig.subplots_adjust(wspace=0.25, hspace=0.15, left=0.05, right=0.95, top=0.95, bottom=0.05)

plt.savefig('/home/marcin/kod/flir-test/data-flir-test/wynik_alternatywny.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved alternative layout with consistent two-line headers.")